# CSE 450 Module 02 — Bank Marketing

**Goal:** predict whether a customer will subscribe to a term deposit (`y`) and evaluate whether model-based targeting improves on random customer selection.

This notebook uses **Polars** for data work and **scikit-learn** for Decision Tree / Random Forest modeling. Decision Tree is the primary model; Random Forest is the comparison model.


In [ ]:
import polars as pl
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

ROOT = Path('..')
MODEL_DIR = ROOT / 'models'
PRED_DIR = ROOT / 'predictions'
MODEL_DIR.mkdir(exist_ok=True)
PRED_DIR.mkdir(exist_ok=True)

TRAIN_URL = 'https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank.csv'
MINI_URL = 'https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank_holdout_test_mini.csv'
HOLDOUT_URL = 'https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank_holdout_test.csv'


## 1. Load and inspect the course data


In [ ]:
df = pl.read_csv(TRAIN_URL)
print(df.shape)
print(df.schema)
df.head()


In [ ]:
df.describe()


## 2. Target distribution and data quality


In [ ]:
target_summary = (df.group_by('y').agg(pl.len().alias('count')).with_columns((pl.col('count')/pl.col('count').sum()).alias('proportion')))
target_summary


In [ ]:
print('Null counts:')
print(df.null_count())

categorical_cols = [name for name, dtype in df.schema.items() if dtype == pl.String]
unknown_counts = df.select([(pl.col(c) == 'unknown').sum().alias(c) for c in categorical_cols])
print('Unknown counts:')
print(unknown_counts)

duplicate_groups = df.group_by(df.columns).len().filter(pl.col('len') > 1)
print('Duplicate groups:', duplicate_groups.height)


In [ ]:
df = df.with_columns(pl.when(pl.col('y') == 'yes').then(1).otherwise(0).alias('target'))
overall_response_rate = df['target'].mean()
print(f'Overall response rate: {overall_response_rate:.4f}')


## 3. Target-focused EDA


In [ ]:
def response_by(frame, column):
    return (frame.group_by(column).agg(
        pl.len().alias('customers'),
        pl.col('target').sum().alias('subscribers'),
        pl.col('target').mean().alias('response_rate')
    ).with_columns((pl.col('response_rate')*100).round(2).alias('response_pct'))
     .sort('response_rate', descending=True))

for col in ['job','marital','education','housing','loan','contact','month','day_of_week','poutcome']:
    print(f'\n--- {col} ---')
    print(response_by(df, col))


In [ ]:
df = df.with_columns(
    pl.when(pl.col('age') < 25).then(pl.lit('<25'))
      .when(pl.col('age') < 35).then(pl.lit('25-34'))
      .when(pl.col('age') < 45).then(pl.lit('35-44'))
      .when(pl.col('age') < 55).then(pl.lit('45-54'))
      .when(pl.col('age') < 65).then(pl.lit('55-64'))
      .otherwise(pl.lit('65+')).alias('age_group'),
    pl.when(pl.col('campaign') == 1).then(pl.lit('1'))
      .when(pl.col('campaign') == 2).then(pl.lit('2'))
      .when(pl.col('campaign') == 3).then(pl.lit('3'))
      .when(pl.col('campaign').is_between(4,5)).then(pl.lit('4-5'))
      .otherwise(pl.lit('6+')).alias('campaign_group'),
    pl.when(pl.col('pdays') == 999).then(pl.lit('never'))
      .when(pl.col('pdays') <= 7).then(pl.lit('0-7 days'))
      .when(pl.col('pdays') <= 30).then(pl.lit('8-30 days'))
      .otherwise(pl.lit('31+ days')).alias('pdays_group')
)
print(response_by(df, 'age_group'))
print(response_by(df, 'campaign_group'))
print(response_by(df, 'previous'))
print(response_by(df, 'pdays_group'))
print(response_by(df, 'poutcome'))


### Timing interactions
The minimum group-size filter reduces the risk of treating tiny segments as actionable patterns.


In [ ]:
job_day = (df.group_by(['job','day_of_week']).agg(pl.len().alias('customers'), pl.col('target').mean().alias('response_rate')).filter(pl.col('customers') >= 100).sort('response_rate', descending=True))
marital_day = (df.group_by(['marital','day_of_week']).agg(pl.len().alias('customers'), pl.col('target').mean().alias('response_rate')).filter(pl.col('customers') >= 100).sort('response_rate', descending=True))
job_month = (df.group_by(['job','month']).agg(pl.len().alias('customers'), pl.col('target').mean().alias('response_rate')).filter(pl.col('customers') >= 100).sort('response_rate', descending=True))
print(job_day.head(20))
print(marital_day.head(20))
print(job_month.head(20))


### Economic variables


In [ ]:
economic_cols = ['emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed']
for col in economic_cols:
    print(f'\n--- {col} ---')
    print(df.group_by(col).agg(pl.len().alias('customers'), pl.col('target').mean().alias('response_rate')).sort(col))


## 4. Reusable feature engineering
Only production-appropriate engineered features belong here. EDA bins are intentionally excluded.


In [ ]:
def engineer_features(frame):
    return frame.with_columns((pl.col('pdays') != 999).cast(pl.Int8).alias('previously_contacted'))

df = engineer_features(df)


## 5. Prepare train/test data


In [ ]:
model_df = df.drop(['age_group','campaign_group','pdays_group'])
X_polars = model_df.drop(['y','target'])
y = model_df['target'].to_numpy()
X = X_polars.to_pandas()

categorical_features = X.select_dtypes(include=['object','string']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object','string']).columns.tolist()

preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('numeric', 'passthrough', numeric_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Train response rate:', y_train.mean())
print('Test response rate:', y_test.mean())


## 6. Baseline


In [ ]:
baseline_pred = np.zeros(len(y_test), dtype=int)
print('Baseline accuracy:', accuracy_score(y_test, baseline_pred))
print('Baseline precision:', precision_score(y_test, baseline_pred, zero_division=0))
print('Baseline recall:', recall_score(y_test, baseline_pred, zero_division=0))
print('Baseline F1:', f1_score(y_test, baseline_pred, zero_division=0))


In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    return {
        'model': name,
        'train_accuracy': accuracy_score(y_train, train_pred),
        'test_accuracy': accuracy_score(y_test, test_pred),
        'precision': precision_score(y_test, test_pred, zero_division=0),
        'recall': recall_score(y_test, test_pred, zero_division=0),
        'f1': f1_score(y_test, test_pred, zero_division=0),
        'customers_contacted': int(test_pred.sum()),
        'subscribers_found': int(((test_pred == 1) & (y_test == 1)).sum())
    }


## 7. Decision Tree — primary model


In [ ]:
tree_full = Pipeline([('preprocess', preprocessor), ('model', DecisionTreeClassifier(random_state=42))])
tree_full.fit(X_train, y_train)
print(evaluate_model('Unrestricted Tree', tree_full, X_train, y_train, X_test, y_test))


In [ ]:
tree_results = []
for depth in [3,5,7,10,15]:
    tree = Pipeline([('preprocess', preprocessor), ('model', DecisionTreeClassifier(max_depth=depth, random_state=42))])
    tree.fit(X_train, y_train)
    tree_results.append(evaluate_model(f'Depth {depth}', tree, X_train, y_train, X_test, y_test))
pl.DataFrame(tree_results)


In [ ]:
leaf_results = []
for leaf_size in [1,5,10,25,50]:
    tree = Pipeline([('preprocess', preprocessor), ('model', DecisionTreeClassifier(max_depth=7, min_samples_leaf=leaf_size, random_state=42))])
    tree.fit(X_train, y_train)
    leaf_results.append(evaluate_model(f'Leaf {leaf_size}', tree, X_train, y_train, X_test, y_test))
pl.DataFrame(leaf_results)


### Select final tree
**Adjust these parameters after reviewing the experiment tables above.** The defaults below are a starting point, not claimed to be optimal.


In [ ]:
FINAL_TREE_DEPTH = 7
FINAL_TREE_MIN_LEAF = 10

final_tree = Pipeline([('preprocess', preprocessor), ('model', DecisionTreeClassifier(max_depth=FINAL_TREE_DEPTH, min_samples_leaf=FINAL_TREE_MIN_LEAF, random_state=42))])
final_tree.fit(X_train, y_train)
final_tree_results = evaluate_model('Final Decision Tree', final_tree, X_train, y_train, X_test, y_test)
print(final_tree_results)

tree_predictions = final_tree.predict(X_test)
print(confusion_matrix(y_test, tree_predictions))
print(classification_report(y_test, tree_predictions))


In [ ]:
feature_names = final_tree.named_steps['preprocess'].get_feature_names_out()
plt.figure(figsize=(24,12))
plot_tree(final_tree.named_steps['model'], feature_names=feature_names, class_names=['No','Yes'], filled=True, max_depth=3, fontsize=8)
plt.show()

tree_importance_df = pl.DataFrame({'feature': feature_names, 'importance': final_tree.named_steps['model'].feature_importances_}).sort('importance', descending=True)
tree_importance_df.head(20)


## 8. Random Forest — comparison model


In [ ]:
random_forest = Pipeline([('preprocess', preprocessor), ('model', RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1))])
random_forest.fit(X_train, y_train)
rf_results = evaluate_model('Random Forest', random_forest, X_train, y_train, X_test, y_test)
comparison = pl.DataFrame([final_tree_results, rf_results])
comparison


## 9. Targeted vs. random marketing


In [ ]:
predictions = final_tree.predict(X_test)
random_response_rate = y_test.mean()
contact_mask = predictions == 1
targeted_response_rate = y_test[contact_mask].mean() if contact_mask.any() else np.nan
lift = targeted_response_rate / random_response_rate if contact_mask.any() else np.nan
print('Random response rate:', random_response_rate)
print('Targeted response rate:', targeted_response_rate)
print('Lift:', lift)


### Optional probability-ranking analysis


In [ ]:
probabilities = final_tree.predict_proba(X_test)[:,1]
ranked = pl.DataFrame({'probability': probabilities, 'actual': y_test}).sort('probability', descending=True)

def lift_at_percent(data, percentage):
    n = max(1, int(data.height * percentage))
    selected = data.head(n)
    targeted_rate = selected['actual'].mean()
    overall_rate = data['actual'].mean()
    return {'contact_pct': int(percentage*100), 'customers_contacted': n, 'response_rate': targeted_rate, 'lift': targeted_rate/overall_rate}

lift_table = pl.DataFrame([lift_at_percent(ranked,p) for p in [0.10,0.20,0.30,0.50]])
lift_table


## 10. Save models
Pickle protocol 5 is used as required by the case study.


In [ ]:
with open(MODEL_DIR / 'decision_tree_model.pkl', 'wb') as f:
    pickle.dump(final_tree, f, protocol=5)
with open(MODEL_DIR / 'random_forest_model.pkl', 'wb') as f:
    pickle.dump(random_forest, f, protocol=5)
print('Saved models to', MODEL_DIR.resolve())


## 11. Mini holdout
Use this to verify the end-to-end prediction pipeline and submission format. Do not repeatedly tune the model to mini-holdout performance.


In [ ]:
mini = engineer_features(pl.read_csv(MINI_URL))
mini_predictions = final_tree.predict(mini.to_pandas())
mini_submission = pl.DataFrame({'y': mini_predictions})
mini_submission.write_csv(PRED_DIR / 'teamX-module2-predictions-mini.csv')
print(mini_submission.shape)
mini_submission.head()


## 12. Final holdout
Do not delete, sort, or shuffle rows. Generate exactly one prediction per original row.


In [ ]:
holdout = engineer_features(pl.read_csv(HOLDOUT_URL))
print('Holdout shape:', holdout.shape)
holdout_predictions = final_tree.predict(holdout.to_pandas())
assert len(holdout_predictions) == holdout.height
assert set(np.unique(holdout_predictions)).issubset({0,1})

submission = pl.DataFrame({'predictions': holdout_predictions})
assert submission.columns == ['predictions']
submission.write_csv(PRED_DIR / 'team_predictions.csv')
print('Submission shape:', submission.shape)
print(submission['predictions'].value_counts())


## 13. Findings to carry into the group executive summary

After running the notebook, record only findings supported by the actual output:

- Overall response rate / class imbalance
- Strongest customer segments with adequate sample sizes
- Association between contact frequency and response (do not claim causation)
- Important timing/customer interactions
- Whether economic conditions materially affect response/model behavior
- Decision Tree vs. Random Forest precision, recall, F1, and train/test gap
- Targeted response rate vs. random response rate and lift
- Final model chosen for holdout predictions and the reason for that choice


## Final package outputs
Run the following cell only after `final_tree` and `random_forest` have been trained. It creates the exact two persisted models and the 4,119-row prediction CSV required by the project.

In [ ]:
# PACKAGE OUTPUTS — run this after the final models are trained
from pathlib import Path
import pickle
import polars as pl
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

models_dir = PROJECT_ROOT / "models"
predictions_dir = PROJECT_ROOT / "predictions"
models_dir.mkdir(exist_ok=True)
predictions_dir.mkdir(exist_ok=True)

# Persist complete fitted pipelines with protocol 5.
with open(models_dir / "decision_tree_model.pkl", "wb") as f:
    pickle.dump(final_tree, f, protocol=5)

with open(models_dir / "random_forest_model.pkl", "wb") as f:
    pickle.dump(random_forest, f, protocol=5)

# Final holdout predictions.
holdout_url = (
    "https://raw.githubusercontent.com/byui-cse/"
    "cse450-course/master/data/bank_holdout_test.csv"
)
holdout = pl.read_csv(holdout_url)
holdout = engineer_features(holdout)
holdout_pd = holdout.to_pandas()

holdout_predictions = final_tree.predict(holdout_pd)

assert len(holdout_predictions) == 4119
assert set(np.unique(holdout_predictions)).issubset({0, 1})

submission = pl.DataFrame({"predictions": holdout_predictions})
assert submission.shape == (4119, 1)
submission.write_csv(predictions_dir / "team_predictions.csv")

print("Created:")
print(models_dir / "decision_tree_model.pkl")
print(models_dir / "random_forest_model.pkl")
print(predictions_dir / "team_predictions.csv")
